# 🧠 PHẦN 2: KHẢO NGHIỆM HỌ KIẾN TRÚC SOTA (VISION TRANSFORMER & DENSE SKIP CNN)
## Đề tài: Phân đoạn và phân loại tổn thương u não trên ảnh cộng hưởng từ (MRI)
**Sinh viên thực hiện:** Trần Mạnh Hùng — **MSSV:** 2351267264 — **Lớp:** S26-65TTNT  
**Giảng viên hướng dẫn:** TS. Hoàng Văn Quý  
**Bộ môn:** Khoa học Dữ liệu và Trí tuệ Nhân tạo — Phân hiệu Trường Đại học Thủy Lợi

---

### 📌 MỤC TIÊU NGHIÊN CỨU TRỌNG TÂM CỦA PHẦN 2:
1. **Khảo sát Năng lực của Vision Transformer trong Phân đoạn Y tế:**
   - Triển khai **SegFormer (Encoder `mit_b2` + All-MLP Decoder)**:
   - Mô hình này đại diện cho trường phái Transformer (tương tự như **Swin-HAFNet** công bố trên *Nature Scientific Data* 2025 của bộ dữ liệu BRISC).
   - Nắm bắt ngữ cảnh không gian toàn cầu (Long-range spatial dependencies) mà các mạng CNN truyền thống bị giới hạn bởi trường cảm nhận cục bộ (Receptive Field).
2. **Khảo nghiệm Kiến trúc Mạng Nối tắt Dày đặc (nnU-Net Style):**
   - Triển khai **ResNeXt-UNet++ (`se_resnext50_32x4d` + Nested Dense Skip Pathways)**:
   - Thu hẹp khoảng cách biểu diễn ngữ nghĩa giữa encoder và decoder, hạn chế suy thoái đặc trưng ranh giới ở các khối u kích thước nhỏ.
3. **Thực nghiệm đối chuẩn chuyên sâu:**
   - Đánh giá khả năng tổng quát hóa, tốc độ suy luận (FPS) và độ chính xác ranh giới (HD95, ASSD) trên toàn bộ các chuỗi xung MRI.


In [1]:
# ── BƯỚC 1: CÀI ĐẶT THƯ VIỆN & THIẾT LẬP PHẦN CỨNG GPU ──
!pip install -q segmentation-models-pytorch albumentations scipy opencv-python reportlab
import os, sys, glob, random, time, gc
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from scipy.spatial import cKDTree

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

seed_everything(42)
print(f'Thiết bị thực thi: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU Model: {torch.cuda.get_device_name(0)}')


Thiết bị thực thi: cuda
GPU Model: Tesla T4


In [2]:
# ── BƯỚC 2: CẤU HÌNH THAM SỐ TOÀN CỤC (HYPERPARAMETERS) ──
CONFIG = {
    'img_size': 256,
    'batch_size': 16,
    'num_classes': 4,
    'epochs': 25,
    'lr': 3e-4,
    'weight_decay': 1e-4,
    'pixel_spacing_mm': 0.5,
    'class_names': {0: 'Background', 1: 'Glioma', 2: 'Meningioma', 3: 'Pituitary'}
}
print('Cấu hình tham số Part 2 đã sẵn sàng!')


Cấu hình tham số Part 2 đã sẵn sàng!


In [3]:
# ── BƯỚC 3: XÁC ĐỊNH ĐƯỜNG DẪN DỮ LIỆU TỰ ĐỘNG TRÊN KAGGLE ──
import zipfile

data_dir = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'UNet' in dirs or ('images' in dirs and 'masks' in dirs):
        data_dir = os.path.join(root, 'UNet') if 'UNet' in dirs else root
        break

if data_dir is None:
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    if zips:
        print(f'Tìm thấy file nén: {zips[0]}, đang giải nén...')
        with zipfile.ZipFile(zips[0], 'r') as z:
            z.extractall('/kaggle/working/dataset')
        data_dir = '/kaggle/working/dataset/UNet' if os.path.exists('/kaggle/working/dataset/UNet') else '/kaggle/working/dataset'

if data_dir is None or not os.path.exists(data_dir):
    data_dir = 'UNet' if os.path.exists('UNet') else '.'

print(f'Thư mục dữ liệu xác định: {data_dir}')

train_img_paths = sorted(glob.glob(f'{data_dir}/images/train/*.jpg') + glob.glob(f'{data_dir}/images/train/*.png'))
train_mask_paths = [p.replace('/images/', '/masks/').replace('.jpg', '.png') for p in train_img_paths]

val_img_paths = sorted(glob.glob(f'{data_dir}/images/val/*.jpg') + glob.glob(f'{data_dir}/images/val/*.png'))
val_mask_paths = [p.replace('/images/', '/masks/').replace('.jpg', '.png') for p in val_img_paths]

print(f'Số lượng mẫu Train: {len(train_img_paths)} | Val: {len(val_img_paths)}')


Thư mục dữ liệu xác định: /kaggle/input/datasets/trnmnhhm/tumor-unet-dataset
Số lượng mẫu Train: 3966 | Val: 693


In [4]:
# ── BƯỚC 4: PIPELINE DỮ LIỆU & BỘ CHỈ SỐ LÂM SÀNG (HD95, ASSD) ──

class BrainTumorDataset(Dataset):
    def __init__(self, img_paths, mask_paths, transform=None):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.img_paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)

        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']

        return img, mask.long()

train_transforms = A.Compose([
    A.Resize(CONFIG['img_size'], CONFIG['img_size']),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transforms = A.Compose([
    A.Resize(CONFIG['img_size'], CONFIG['img_size']),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

train_loader = DataLoader(BrainTumorDataset(train_img_paths, train_mask_paths, train_transforms), batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(BrainTumorDataset(val_img_paths, val_mask_paths, val_transforms), batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

def compute_hd95_assd(pred_binary, gt_binary, spacing_mm=CONFIG['pixel_spacing_mm']):
    contours_p, _ = cv2.findContours(pred_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    contours_g, _ = cv2.findContours(gt_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    pts_p = np.vstack([c.squeeze(axis=1) for c in contours_p if len(c) > 0]) if contours_p else np.empty((0, 2))
    pts_g = np.vstack([c.squeeze(axis=1) for c in contours_g if len(c) > 0]) if contours_g else np.empty((0, 2))

    if len(pts_p) == 0 and len(pts_g) == 0:
        return 0.0, 0.0
    if len(pts_p) == 0 or len(pts_g) == 0:
        max_penalty = np.sqrt(2 * (CONFIG['img_size'] ** 2)) * spacing_mm
        return max_penalty, max_penalty

    pts_p = pts_p * spacing_mm
    pts_g = pts_g * spacing_mm

    tree_g = cKDTree(pts_g)
    tree_p = cKDTree(pts_p)

    d_p2g, _ = tree_g.query(pts_p)
    d_g2p, _ = tree_p.query(pts_g)

    all_d = np.concatenate([d_p2g, d_g2p])
    hd95 = float(np.percentile(all_d, 95))
    assd = float((np.mean(d_p2g) + np.mean(d_g2p)) / 2.0)
    return hd95, assd

def evaluate_model(model, dataloader):
    model.eval()
    dice_scores, iou_scores, hd95_scores, assd_scores = [], [], [], []
    latencies = []

    with torch.no_grad():
        for imgs, masks in dataloader:
            imgs = imgs.to(DEVICE)
            t0 = time.time()
            outputs = model(imgs)
            if DEVICE.type == 'cuda': torch.cuda.synchronize()
            latencies.append((time.time() - t0) / imgs.size(0))

            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            gts = masks.numpy()

            for b in range(imgs.size(0)):
                for c in [1, 2, 3]:
                    p_c = (preds[b] == c)
                    g_c = (gts[b] == c)
                    if g_c.any() or p_c.any():
                        inter = np.logical_and(p_c, g_c).sum()
                        union = p_c.sum() + g_c.sum()
                        dice = (2.0 * inter + 1e-6) / (union + 1e-6)
                        iou = (inter + 1e-6) / (p_c.sum() + g_c.sum() - inter + 1e-6)
                        hd95, assd = compute_hd95_assd(p_c, g_c)

                        dice_scores.append(dice)
                        iou_scores.append(iou)
                        hd95_scores.append(hd95)
                        assd_scores.append(assd)

    return {
        'dice': float(np.mean(dice_scores)),
        'iou': float(np.mean(iou_scores)),
        'hd95_mm': float(np.mean(hd95_scores)),
        'assd_mm': float(np.mean(assd_scores)),
        'fps': float(1.0 / np.mean(latencies))
    }


/usr/local/lib/python3.13/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [5]:
# ── BƯỚC 5: HUẤN LUYỆN MÔ HÌNH 3: RESNEXT-UNET++ (NESTED DENSE SKIP PATHWAYS) ──
print('\n' + '='*60)
print('🚀 [MÔ HÌNH 3/4] BẮT ĐẦU HUẤN LUYỆN RESNEXT-UNET++ (DENSE SKIP CNN - nnU-Net STYLE)')
print('='*60)

model_unetplus = smp.UnetPlusPlus(
    encoder_name='se_resnext50_32x4d',
    encoder_weights='imagenet',
    in_channels=3,
    classes=CONFIG['num_classes'],
    decoder_attention_type='scse',
    activation=None
).to(DEVICE)

loss_fn = smp.losses.DiceLoss(mode='multiclass', from_logits=True)
ce_fn = nn.CrossEntropyLoss()
def combo_loss(p, t): return 0.5 * loss_fn(p, t) + 0.5 * ce_fn(p, t)

optimizer_pp = torch.optim.AdamW(model_unetplus.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler_pp = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_pp, T_max=CONFIG['epochs'], eta_min=1e-6)
scaler_pp = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))

best_dice_pp = 0.0
for epoch in range(1, CONFIG['epochs'] + 1):
    model_unetplus.train()
    total_loss = 0.0
    for imgs, targets in train_loader:
        imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
        optimizer_pp.zero_grad()
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
            logits = model_unetplus(imgs)
            loss = combo_loss(logits, targets)
        scaler_pp.scale(loss).backward()
        scaler_pp.step(optimizer_pp)
        scaler_pp.update()
        total_loss += loss.item()

    scheduler_pp.step()
    val_res = evaluate_model(model_unetplus, val_loader)
    print(f'Epoch [{epoch:02d}/{CONFIG["epochs"]:02d}] | Loss: {total_loss/len(train_loader):.4f} | Val Dice: {val_res["dice"]:.4f} | HD95: {val_res["hd95_mm"]:.2f}mm')

    if val_res['dice'] > best_dice_pp:
        best_dice_pp = val_res['dice']
        torch.save(model_unetplus.state_dict(), '/kaggle/working/best_resunet_plusplus.pt')

print(f'✅ Hoàn tất Mô hình 3! Best Dice: {best_dice_pp:.4f}')

del model_unetplus, optimizer_pp, scheduler_pp
torch.cuda.empty_cache()
gc.collect()



🚀 [MÔ HÌNH 3/4] BẮT ĐẦU HUẤN LUYỆN RESNEXT-UNET++ (DENSE SKIP CNN - nnU-Net STYLE)


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  111MB            

model.safetensors: downloading bytes:           |  0.00B            

/tmp/ipykernel_23/1360233966.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_pp = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))
/tmp/ipykernel_23/1360233966.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):


Epoch [01/25] | Loss: 0.3178 | Val Dice: 0.6829 | HD95: 26.80mm
Epoch [02/25] | Loss: 0.1227 | Val Dice: 0.6812 | HD95: 31.29mm
Epoch [03/25] | Loss: 0.1084 | Val Dice: 0.6744 | HD95: 33.79mm
Epoch [04/25] | Loss: 0.1009 | Val Dice: 0.7421 | HD95: 19.93mm
Epoch [05/25] | Loss: 0.0950 | Val Dice: 0.6901 | HD95: 30.87mm
Epoch [06/25] | Loss: 0.0884 | Val Dice: 0.7379 | HD95: 21.55mm
Epoch [07/25] | Loss: 0.0894 | Val Dice: 0.7267 | HD95: 26.56mm
Epoch [08/25] | Loss: 0.0827 | Val Dice: 0.7839 | HD95: 13.44mm
Epoch [09/25] | Loss: 0.0771 | Val Dice: 0.7563 | HD95: 19.95mm
Epoch [10/25] | Loss: 0.0772 | Val Dice: 0.7847 | HD95: 15.10mm
Epoch [11/25] | Loss: 0.0716 | Val Dice: 0.7680 | HD95: 18.79mm
Epoch [12/25] | Loss: 0.0733 | Val Dice: 0.7041 | HD95: 30.76mm
Epoch [13/25] | Loss: 0.0700 | Val Dice: 0.8009 | HD95: 12.97mm
Epoch [14/25] | Loss: 0.0671 | Val Dice: 0.8077 | HD95: 11.21mm
Epoch [15/25] | Loss: 0.0650 | Val Dice: 0.8129 | HD95: 10.81mm
Epoch [16/25] | Loss: 0.0630 | Val Dice:

48

In [6]:
# ── BƯỚC 6: HUẤN LUYỆN MÔ HÌNH 4: SEGFORMER (VISION TRANSFORMER MIT-B2) ──
print('\n' + '='*60)
print('🚀 [MÔ HÌNH 4/4] BẮT ĐẦU HUẤN LUYỆN SEGFORMER (VISION TRANSFORMER - Swin-HAFNet STYLE)')
print('='*60)

model_transformer = smp.Segformer(
    encoder_name='mit_b2',
    encoder_weights='imagenet',
    in_channels=3,
    classes=CONFIG['num_classes'],
    activation=None
).to(DEVICE)

optimizer_tf = torch.optim.AdamW(model_transformer.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler_tf = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_tf, T_max=CONFIG['epochs'], eta_min=1e-6)
scaler_tf = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))

best_dice_tf = 0.0
for epoch in range(1, CONFIG['epochs'] + 1):
    model_transformer.train()
    total_loss = 0.0
    for imgs, targets in train_loader:
        imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
        optimizer_tf.zero_grad()
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
            logits = model_transformer(imgs)
            loss = combo_loss(logits, targets)
        scaler_tf.scale(loss).backward()
        scaler_tf.step(optimizer_tf)
        scaler_tf.update()
        total_loss += loss.item()

    scheduler_tf.step()
    val_res = evaluate_model(model_transformer, val_loader)
    print(f'Epoch [{epoch:02d}/{CONFIG["epochs"]:02d}] | Loss: {total_loss/len(train_loader):.4f} | Val Dice: {val_res["dice"]:.4f} | HD95: {val_res["hd95_mm"]:.2f}mm')

    if val_res['dice'] > best_dice_tf:
        best_dice_tf = val_res['dice']
        torch.save(model_transformer.state_dict(), '/kaggle/working/best_segformer_transformer.pt')

print(f'✅ Hoàn tất Mô hình 4! Best Dice: {best_dice_tf:.4f}')



🚀 [MÔ HÌNH 4/4] BẮT ĐẦU HUẤN LUYỆN SEGFORMER (VISION TRANSFORMER - Swin-HAFNet STYLE)


config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 98.9MB            

model.safetensors: downloading bytes:           |  0.00B            

/tmp/ipykernel_23/1466207197.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_tf = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))
/tmp/ipykernel_23/1466207197.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):


Epoch [01/25] | Loss: 0.2843 | Val Dice: 0.5806 | HD95: 41.48mm
Epoch [02/25] | Loss: 0.1287 | Val Dice: 0.6947 | HD95: 25.16mm
Epoch [03/25] | Loss: 0.1145 | Val Dice: 0.5933 | HD95: 45.15mm
Epoch [04/25] | Loss: 0.0972 | Val Dice: 0.7398 | HD95: 18.65mm
Epoch [05/25] | Loss: 0.0936 | Val Dice: 0.7433 | HD95: 20.53mm
Epoch [06/25] | Loss: 0.0904 | Val Dice: 0.7272 | HD95: 19.97mm
Epoch [07/25] | Loss: 0.0865 | Val Dice: 0.7331 | HD95: 22.35mm
Epoch [08/25] | Loss: 0.0837 | Val Dice: 0.6998 | HD95: 26.59mm
Epoch [09/25] | Loss: 0.0827 | Val Dice: 0.7791 | HD95: 12.79mm
Epoch [10/25] | Loss: 0.0779 | Val Dice: 0.7832 | HD95: 12.92mm
Epoch [11/25] | Loss: 0.0733 | Val Dice: 0.7897 | HD95: 12.87mm
Epoch [12/25] | Loss: 0.0748 | Val Dice: 0.8058 | HD95: 10.45mm
Epoch [13/25] | Loss: 0.0732 | Val Dice: 0.7791 | HD95: 16.85mm
Epoch [14/25] | Loss: 0.0719 | Val Dice: 0.7651 | HD95: 16.84mm
Epoch [15/25] | Loss: 0.0664 | Val Dice: 0.7908 | HD95: 12.79mm
Epoch [16/25] | Loss: 0.0624 | Val Dice:

In [7]:
# ── BƯỚC 7: ĐỐI CHUẨN HIỆU NĂNG GIỮA RESNEXT-UNET++ VÀ SEGFORMER TRANSFORMER ──

m_pp = smp.UnetPlusPlus('se_resnext50_32x4d', in_channels=3, classes=CONFIG['num_classes'], decoder_attention_type='scse').to(DEVICE)
m_pp.load_state_dict(torch.load('/kaggle/working/best_resunet_plusplus.pt'))

m_tf = smp.Segformer('mit_b2', in_channels=3, classes=CONFIG['num_classes']).to(DEVICE)
m_tf.load_state_dict(torch.load('/kaggle/working/best_segformer_transformer.pt'))

res_pp = evaluate_model(m_pp, val_loader)
res_tf = evaluate_model(m_tf, val_loader)

df_part2 = pd.DataFrame([
    {
        'Kiến trúc Mô hình': 'ResNeXt-UNet++ (nnU-Net Style)',
        'Họ thuật toán': 'Dense Skip Pathways CNN',
        'Dice Score (DSC)': f"{res_pp['dice']:.4f}",
        'IoU (%)': f"{res_pp['iou']*100:.2f}%",
        'Sai lệch viền HD95 (mm)': f"{res_pp['hd95_mm']:.2f} mm",
        'ASSD (mm)': f"{res_pp['assd_mm']:.2f} mm",
        'Tốc độ (FPS)': f"{res_pp['fps']:.1f}",
        'Nhận xét': 'Bảo toàn độ phân giải không gian rất tốt, các nhánh viền sắc nét'
    },
    {
        'Kiến trúc Mô hình': 'SegFormer (Vision Transformer)',
        'Họ thuật toán': 'Self-Attention + All-MLP Decoder',
        'Dice Score (DSC)': f"{res_tf['dice']:.4f}",
        'IoU (%)': f"{res_tf['iou']*100:.2f}%",
        'Sai lệch viền HD95 (mm)': f"{res_tf['hd95_mm']:.2f} mm",
        'ASSD (mm)': f"{res_tf['assd_mm']:.2f} mm",
        'Tốc độ (FPS)': f"{res_tf['fps']:.1f}",
        'Nhận xét': 'Đạt độ chính xác cao nhất: nắm bắt tương quan giải phẫu toàn cục cực mạnh'
    }
])

print('========================================================================================')
print('📊 KẾT QUẢ ĐỐI CHUẨN PART 2: RESNEXT-UNET++ VS VISION TRANSFORMER (SEGFORMER)')
print('========================================================================================')
display(df_part2)
df_part2.to_csv('/kaggle/working/benchmark_part2_transformer_results.csv', index=False)
print('✅ Đã lưu kết quả đối chuẩn tại /kaggle/working/benchmark_part2_transformer_results.csv')


📊 KẾT QUẢ ĐỐI CHUẨN PART 2: RESNEXT-UNET++ VS VISION TRANSFORMER (SEGFORMER)


,Kiến trúc Mô hình,Họ thuật toán,Dice Score (DSC),IoU (%),Sai lệch viền HD95 (mm),ASSD (mm),Tốc độ (FPS),Nhận xét
0,ResNeXt-UNet++ (nnU-Net Style),Dense Skip Pathways CNN,0.8277,74.60%,9.03 mm,6.83 mm,36.3,"Bảo toàn độ phân giải không gian rất tốt, các ..."
1,SegFormer (Vision Transformer),Self-Attention + All-MLP Decoder,0.8300,74.50%,7.63 mm,5.58 mm,163.6,Đạt độ chính xác cao nhất: nắm bắt tương quan ...


✅ Đã lưu kết quả đối chuẩn tại /kaggle/working/benchmark_part2_transformer_results.csv
